# DEH 30-Day PySpark Challenge
## Day 23 — Practice Set 2: Joins

**Instructions:**
1. Click `File → Save a copy in Drive` before editing
2. For each problem: write SQL first, then PySpark, in the empty cells
3. Reference solutions follow each problem — try first before checking

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder
    .appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

In [ ]:
orders_schema = StructType([
    StructField("order_id",       StringType(),  True),
    StructField("customer_id",    StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("order_date",     DateType(),    True),
    StructField("quantity",       IntegerType(), True),
    StructField("unit_price",     DoubleType(),  True),
    StructField("discount_pct",   IntegerType(), True),
    StructField("status",         StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("region",         StringType(),  True)
])

customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name",  StringType(), True),
    StructField("last_name",   StringType(), True),
    StructField("email",       StringType(), True),
    StructField("city",        StringType(), True),
    StructField("state",       StringType(), True),
    StructField("country",     StringType(), True),
    StructField("signup_date", DateType(),   True),
    StructField("segment",     StringType(), True)
])

products_schema = StructType([
    StructField("product_id",     StringType(), True),
    StructField("product_name",   StringType(), True),
    StructField("category",       StringType(), True),
    StructField("sub_category",   StringType(), True),
    StructField("unit_price",     DoubleType(), True),
    StructField("cost_price",     DoubleType(), True),
    StructField("supplier",       StringType(), True),
    StructField("stock_quantity", IntegerType(),True)
])

orders_df    = spark.read.option("header","true").schema(orders_schema).csv(f"s3a://{BUCKET}/data/orders.csv")
customers_df = spark.read.option("header","true").schema(customers_schema).csv(f"s3a://{BUCKET}/data/customers.csv")
products_df  = spark.read.option("header","true").schema(products_schema).csv(f"s3a://{BUCKET}/data/products.csv")

orders_df.createOrReplaceTempView("orders")
customers_df.createOrReplaceTempView("customers")
products_df.createOrReplaceTempView("products")

print(f"Orders: {orders_df.count()} | Customers: {customers_df.count()} | Products: {products_df.count()}")

---
## Problem 1 (Easy) — Orders with customer names

Return `order_id`, `first_name`, `last_name`, `unit_price` for every order joined with its customer. Sort by `unit_price` descending, top 10.

In [ ]:
# Your SQL attempt


In [ ]:
# Your PySpark attempt


### Reference Solution — Problem 1

In [ ]:
spark.sql("""
    SELECT o.order_id, c.first_name, c.last_name, o.unit_price
    FROM orders o
    INNER JOIN customers c ON o.customer_id = c.customer_id
    ORDER BY o.unit_price DESC
    LIMIT 10
""").show()

In [ ]:
orders_df.join(customers_df, on="customer_id", how="inner") \
    .select("order_id", "first_name", "last_name", "unit_price") \
    .orderBy(F.col("unit_price").desc()) \
    .show(10)

---
## Problem 2 (Easy) — Orders with product details

Return `order_id`, `product_name`, `category`, `quantity`. Handle the `unit_price` collision between orders and products.

In [ ]:
# Your SQL attempt


In [ ]:
# Your PySpark attempt


### Reference Solution — Problem 2

In [ ]:
spark.sql("""
    SELECT o.order_id, p.product_name, p.category, o.quantity
    FROM orders o
    INNER JOIN products p ON o.product_id = p.product_id
    LIMIT 10
""").show()

In [ ]:
orders_df.join(products_df, on="product_id", how="inner") \
    .select(
        orders_df["order_id"],
        products_df["product_name"],
        products_df["category"],
        orders_df["quantity"]
    ).show(10)

---
## Problem 3 (Medium) — Customers who have never ordered

Find customers with zero orders. Return `customer_id`, `first_name`, `last_name`.

In [ ]:
# Your SQL attempt


In [ ]:
# Your PySpark attempt


### Reference Solution — Problem 3

In [ ]:
spark.sql("""
    SELECT c.customer_id, c.first_name, c.last_name
    FROM customers c
    LEFT ANTI JOIN orders o ON c.customer_id = o.customer_id
""").show()

In [ ]:
customers_df.join(orders_df, on="customer_id", how="left_anti") \
    .select("customer_id", "first_name", "last_name") \
    .show()

---
## Problem 4 (Medium) — Products that have never been ordered

Find products that don't appear in any order. Return `product_id`, `product_name`, `category`.

In [ ]:
# Your SQL attempt


In [ ]:
# Your PySpark attempt


### Reference Solution — Problem 4

In [ ]:
spark.sql("""
    SELECT p.product_id, p.product_name, p.category
    FROM products p
    LEFT ANTI JOIN orders o ON p.product_id = o.product_id
""").show()

In [ ]:
products_df.join(orders_df, on="product_id", how="left_anti") \
    .select("product_id", "product_name", "category") \
    .show()

---
## Problem 5 (Hard) — Three-way join with aggregation

Join orders + customers + products. Revenue per segment + category combination. Top 10 by revenue.

In [ ]:
# Your SQL attempt


In [ ]:
# Your PySpark attempt


### Reference Solution — Problem 5

In [ ]:
spark.sql("""
    SELECT
        c.segment,
        p.category,
        ROUND(SUM(o.unit_price * o.quantity), 2) AS total_revenue
    FROM orders o
    INNER JOIN customers c ON o.customer_id = c.customer_id
    INNER JOIN products  p ON o.product_id  = p.product_id
    GROUP BY c.segment, p.category
    ORDER BY total_revenue DESC
    LIMIT 10
""").show()

In [ ]:
orders_df.join(customers_df, on="customer_id", how="inner") \
    .join(products_df, on="product_id", how="inner") \
    .groupBy("segment", "category").agg(
        F.round(F.sum(orders_df["unit_price"] * orders_df["quantity"]), 2).alias("total_revenue")
    ).orderBy(F.col("total_revenue").desc()) \
    .show(10)

---
## Problem 6 (Hard) — Self-referencing comparison — customers in the same city

For every pair of different customers in the same city, return both IDs and the city — no duplicate pairs.

In [ ]:
# Your SQL attempt


In [ ]:
# Your PySpark attempt


### Reference Solution — Problem 6

The key trick: join the table to itself on `city`, then use `a.customer_id < b.customer_id` to both exclude self-matches AND prevent duplicate pairs (A-B and B-A).

In [ ]:
spark.sql("""
    SELECT a.customer_id AS customer_1, b.customer_id AS customer_2, a.city
    FROM customers a
    INNER JOIN customers b
        ON a.city = b.city
        AND a.customer_id < b.customer_id
    ORDER BY a.city
""").show()

In [ ]:
# Self-join using aliased DataFrames
a = customers_df.alias("a")
b = customers_df.alias("b")

a.join(
    b,
    on=(F.col("a.city") == F.col("b.city")) & (F.col("a.customer_id") < F.col("b.customer_id")),
    how="inner"
).select(
    F.col("a.customer_id").alias("customer_1"),
    F.col("b.customer_id").alias("customer_2"),
    F.col("a.city")
).orderBy("city").show()

---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*